---
# PART 1 — Raw Implementation (No LangChain, No LangGraph)

We use only the `openai` SDK. This section builds, in order: a single tool call, a full request/response inspection, multi-tool dispatch, parallel tool calls, a general-purpose ReAct loop, Pydantic argument validation, tool-level error handling, forced tool choice, and streaming with delta accumulation.


## 0. Setup

Install dependencies and configure your OpenAI API key. We pin nothing exotic — just the official SDK for Part 1, and `langchain`, `langchain-openai`, `langgraph` for Part 2.


In [ ]:
# %pip install --quiet openai langchain langchain-openai langgraph pydantic


In [1]:
import os
import json
import time
import asyncio
from getpass import getpass

# Set your key as an environment variable. Never hardcode API keys in notebooks you'll share.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OPENAI_API_KEY: ")

MODEL = "gpt-4.1"  # any tool-calling-capable OpenAI model works (gpt-4.1, gpt-4o, gpt-4o-mini, ...)


Enter your OPENAI_API_KEY: ··········


## 1.1 The OpenAI Client

Nothing agent-specific here — a plain client, same as any other OpenAI call.


In [2]:
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from the environment


## 1.2 Defining a Tool Schema by Hand

A tool is a plain Python dict following OpenAI's `tools` format: `type`, `function.name`, `function.description`, `function.parameters` (JSON Schema). Notice how much of this is **prompt engineering** — the `description` fields are read by the model, not just by your type checker.

We define two tools: `get_weather` (simulated — no real API call, to keep the notebook runnable offline) and a `calculator` with basic arithmetic.


In [3]:
def get_weather(city: str, unit: str="celsius") -> dict:
  """Simulated weather lookup - replace with a real API call in production."""
  fake_db = {
      "hanoi": 32,
      "tokyo": 24,
      "san fracisco": 18,
      "new york": 21,
      "london": 15,
  }
  temp_c = fake_db.get(city.lower(), 20)
  temp = temp_c if unit == "celsius" else temp_c * 9 / 5 + 32
  return {"city": city, "temperature": round(temp, 1), "unit": unit, "condition": "sunny"}

def calculator(operation: str, a: float, b: float) -> dict:
  """Basic arithmetic tool."""
  ops = {
      "add": lambda x,y: x+y,
      "substract": lambda x,y: x-y,
      "multiply": lambda x,y: x*y,
      "divide": lambda x,y: x/y if y !=0 else None,
  }
  if operation not in ops:
    raise ValueError(f"Unsupported operation: {operation}")
  result = ops[operation](a,b)
  if result is None:
    raise ZeroDivisionError("Division by zero")
  return {"operation": operation, "a": a, "b": b, "result": result}

# Dispatch table: tool name (string, as the model will send it) -> actual Python callable
TOOL_REGISTRY = {
    "get_weather": get_weather,
    "calculator": calculator,
}

In [4]:
# JSON Schema tool definitions — this is exactly what gets sent to the API in the `tools` field.
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": (
                "Get the current weather for a given city. Use this whenever the user asks "
                "about weather, temperature, or whether they need an umbrella."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name, e.g. 'Hanoi' or 'San Francisco'",
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit to return. Defaults to celsius.",
                    },
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Perform a basic arithmetic operation: add, subtract, multiply, or divide.",
            "parameters": {
                "type": "object",
                "properties": {
                    "operation": {
                        "type": "string",
                        "enum": ["add", "subtract", "multiply", "divide"],
                    },
                    "a": {"type": "number", "description": "First operand"},
                    "b": {"type": "number", "description": "Second operand"},
                },
                "required": ["operation", "a", "b"],
            },
        },
    },
]
print(json.dumps(TOOLS, indent=2))


[
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "Get the current weather for a given city. Use this whenever the user asks about weather, temperature, or whether they need an umbrella.",
      "parameters": {
        "type": "object",
        "properties": {
          "city": {
            "type": "string",
            "description": "The city name, e.g. 'Hanoi' or 'San Francisco'"
          },
          "unit": {
            "type": "string",
            "enum": [
              "celsius",
              "fahrenheit"
            ],
            "description": "Temperature unit to return. Defaults to celsius."
          }
        },
        "required": [
          "city"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "calculator",
      "description": "Perform a basic arithmetic operation: add, subtract, multiply, or divide.",
      "parameters": {
        "type": "object",
        "properties": {


## 1.3 A Single Tool-Calling Turn

Send a message that should trigger `get_weather`. Inspect the raw `message.tool_calls` structure before doing anything else — this is the object every framework in Part 2 ultimately wraps.


In [13]:
messages = [
    {
        "role": "user",
        "content": "What's the weather like in Hanoi right now ?"
    }
]
response = client.chat.completions.create(
    model = MODEL, messages=messages, tools=TOOLS
)
message = response.choices[0].message
print(message)
print(response)
print("content:", message.content)          # usually None on a tool-calling turn
print("tool_calls:", message.tool_calls) # tool_calls: [ChatCompletionMessageFunctionToolCall(id='call_TLzFEDqh8ubmUU7wEaLMecRA', function=Function(arguments='{"city":"Hanoi"}', name='get_weather'), type='function')]


ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_Nn0lcVIksyhLcC74h1u0xGeN', function=Function(arguments='{"city":"Hanoi"}', name='get_weather'), type='function')])
ChatCompletion(id='chatcmpl-Dx8WRjdMQWbVwsfucQEtclODGwCvF', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_Nn0lcVIksyhLcC74h1u0xGeN', function=Function(arguments='{"city":"Hanoi"}', name='get_weather'), type='function')]))], created=1782986483, model='gpt-4.1-2025-04-14', object='chat.completion', moderation=None, service_tier='default', system_fingerprint='fp_2cd7932764', usage=CompletionUsage(completion_tokens=15, prompt_tokens=167, total_tokens=182, completion_tokens_details=CompletionTokensDetails

Notice: `message.content` is typically `None`, and `message.tool_calls` is a list of objects with `id`, `function.name`, and `function.arguments` — the last one a **JSON string**, not a parsed dict. The API never executes your function; it only ever describes intent.


## 1.4 Executing the Tool and Returning the Result

Three steps: parse the arguments, run the real Python function, append a `role="tool"` message tagged with the matching `tool_call_id`. Then call the API again so the model can turn the result into a natural-language answer.


In [15]:
tool_call = message.tool_calls[0]
args = json.loads(tool_call.function.arguments)
print("Parsed args:", args)

result = TOOL_REGISTRY[tool_call.function.name](**args)
print(message.model_dump(exclude_none=True))
messages.append(message.model_dump(exclude_none=True))
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(result)
})
final = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS)
print(final.choices[0].message.content)


Parsed args: {'city': 'Hanoi'}
{'role': 'assistant', 'annotations': [], 'tool_calls': [{'id': 'call_Nn0lcVIksyhLcC74h1u0xGeN', 'function': {'arguments': '{"city":"Hanoi"}', 'name': 'get_weather'}, 'type': 'function'}]}
The weather in Hanoi right now is sunny with a temperature of 32°C. It's quite warm and clear!


## 1.5 Parallel Tool Calls

A single user message can trigger multiple tool calls in one turn (e.g., asking about two cities). `message.tool_calls` is simply a longer list — execute every entry and append one `role="tool"` message per call, in any order, before the next assistant turn.


In [19]:
messages = [
    {
        "role": "user",
        "content": "What's the weather in Hanoi and in Tokyo ? Also, what's 12 multiplied by 7 ?"
    }
]
response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS)
message = response.choices[0].message
print(f"Model requested {len(message.tool_calls)} tool call(s):")
for tc in message.tool_calls:
  print(" -", tc.function.name, tc.function.arguments)

messages.append(message.model_dump(exclude_none=True))
for tool_call in message.tool_calls:
  args = json.loads(tool_call.function.arguments)
  result = TOOL_REGISTRY[tool_call.function.name](**args)
  messages.append({
      "role":"tool",
      "tool_call_id": tool_call.id,
      "content": json.dumps(result)
  })

final = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS)
print("\nFinal answer:\n", final.choices[0].message.content)

Model requested 3 tool call(s):
 - get_weather {"city": "Hanoi"}
 - get_weather {"city": "Tokyo"}
 - calculator {"operation": "multiply", "a": 12, "b": 7}

Final answer:
 Here are the answers to your questions:

- The weather in Hanoi is sunny with a temperature of 32°C.
- The weather in Tokyo is sunny with a temperature of 24°C.
- 12 multiplied by 7 equals 84.


To force strictly sequential calls (e.g., tool B depends on a side effect of tool A), pass `parallel_tool_calls=False` to `chat.completions.create`.


## 1.6 The ReAct Loop, Implemented by Hand

This is the general-purpose function every example above was a hand-unrolled version of. It keeps calling the API and executing tools until the model stops requesting tool calls, or a safety cap is hit. **This exact loop is what `AgentExecutor` and LangGraph's `create_react_agent` automate in Part 2** — understanding it here is the whole point of Phase 2 of the roadmap.


In [22]:
def execute_tool(tool_call):
  """
  Execute one tool call, never letting an exception escape the loop
  """
  try:
    args = json.loads(tool_call.function.arguments)
    return TOOL_REGISTRY[tool_call.function.name](**args)
  except Exception as e:
    return {"error": str(e), "tool": tool_call.function.name}

def run_agent(user_message, max_iterations=8, verbose=True):
  messages = [{
      "role": "user",
      "content": user_message
  }]
  for step in range(1, max_iterations+1):
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS
    )
    msg = response.choices[0].message
    messages.append(msg.model_dump(exclude_none=True))
    if not msg.tool_calls:
      if verbose:
        print(f"[step {step}] Final answer.")
      return msg.content

    for tool_call in msg.tool_calls:
      if verbose:
        print(f"[step {step}] Calling {tool_call.function.name}({tool_call.function.arguments})")
      result = execute_tool(tool_call)
      if verbose:
        print(f"[step {step}] -> {result}")
      messages.append({
          "role": "tool",
          "tool_call_id": tool_call.id,
          "content": json.dumps(result),
      })
  return "Max Iterrations reached without a final answer"

answer = run_agent("What's 8 times 9, and what's the weather in London?")
print("\n=== ANSWER ===\n", answer)


[step 1] Calling calculator({"operation": "multiply", "a": 8, "b": 9})
[step 1] -> {'operation': 'multiply', 'a': 8, 'b': 9, 'result': 72}
[step 1] Calling get_weather({"city": "London"})
[step 1] -> {'city': 'London', 'temperature': 15, 'unit': 'celsius', 'condition': 'sunny'}
[step 2] Final answer.

=== ANSWER ===
 8 times 9 is 72.

The current weather in London is sunny with a temperature of 15°C.


## 1.7 Validating Arguments with Pydantic

Model-generated JSON is untrusted input. Validate it against a schema before executing anything with side effects. On failure, **feed the error back to the model as the tool result** rather than crashing — this lets the model self-correct on the next turn.


In [23]:
from pydantic import BaseModel, Field, ValidationError

class CalculatorArgs(BaseModel):
  operation: str = Field(pattern="^(add|substract|multiply|divide)$")
  a: float
  b: float

def execute_tool_validated(tool_call):
  try:
    if tool_call.function.name == "calculator":
      args = CalculatorArgs.model_validate_json(tool_call.function.arguments)
      return calculator(**args.model_dump())
    args = json.loads(tool_call.function.arguments)
    return TOOL_REGISTRY[tool_call.function.name](**args)
  except (ValidationError, json.JSONDecodeError) as e:
    # Structured error, fed back as an observation -- the model can retry with corrected args.
    return {
        "error": f"Invalid arguments, please retry: {e}"
    }
  except Exception as e:
    return {"error": str(e)}

# Simulate a malformed tool call to see the self-correction path
class FakeToolCall:
    class function:
        name = "calculator"
        arguments = '{"operation": "power", "a": 2, "b": 3}'  # "power" is not a supported operation

print(execute_tool_validated(FakeToolCall))

{'error': "Invalid arguments, please retry: 1 validation error for CalculatorArgs\noperation\n  String should match pattern '^(add|substract|multiply|divide)$' [type=string_pattern_mismatch, input_value='power', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.13/v/string_pattern_mismatch"}


## 1.8 Error Handling Inside Tools

Tools fail for mundane reasons: division by zero, timeouts, missing records. The rule is the same as section 1.7 — **never let an exception kill the agent loop.** Catch it, return a structured error, let the model decide what to do next (retry, try another tool, or apologize to the user).


In [24]:
answer = run_agent("What is 10 divided by 0? If that fails, just tell me it's undefined.")
print("\n=== ANSWER ===\n", answer)


[step 1] Calling calculator({"operation":"divide","a":10,"b":0})
[step 1] -> {'error': 'Division by zero', 'tool': 'calculator'}
[step 2] Final answer.

=== ANSWER ===
 Division by zero is undefined. You cannot divide 10 (or any number) by 0 in mathematics.


## 1.9 Forcing / Restricting Tool Choice

`tool_choice` controls how much freedom the model has this turn:

| Value | Behavior |
|---|---|
| `"auto"` (default) | Model decides whether to call a tool |
| `"required"` | Model must call at least one tool |
| `"none"` | Model must not call a tool |
| `{"type": "function", "function": {"name": "..."}}` | Model must call this specific tool |


In [25]:
# Force the model to always use the calculator tool, even for a question that doesn't need it,
# useful for guided workflows where a specific step always calls a specific tool.
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Just say hello to me."}],
    tools=TOOLS,
    tool_choice={"type": "function", "function": {"name": "calculator"}},
)
print(response.choices[0].message.tool_calls)


[ChatCompletionMessageFunctionToolCall(id='call_0JGXwB0EcHJdS7tWYPXZ71AZ', function=Function(arguments='{"operation":"add","a":2,"b":2}', name='calculator'), type='function')]


## 1.10 Streaming with Tool Calls (Delta Accumulation)

With `stream=True`, tool call fragments arrive across multiple chunks and must be accumulated **by index** before you have a complete, parseable call. This is the trickiest part of a raw implementation, and exactly what LangGraph's `astream_events` hides from you in Part 2.


In [26]:
def stream_agent_turn(messages):
    stream = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, stream=True)

    collected_tool_calls = {}   # index -> accumulating dict
    content_chunks = []

    for chunk in stream:
        delta = chunk.choices[0].delta
        if delta.content:
            content_chunks.append(delta.content)
        if delta.tool_calls:
            for tc_delta in delta.tool_calls:
                idx = tc_delta.index
                if idx not in collected_tool_calls:
                    collected_tool_calls[idx] = {"id": "", "name": "", "arguments": ""}
                if tc_delta.id:
                    collected_tool_calls[idx]["id"] += tc_delta.id
                if tc_delta.function and tc_delta.function.name:
                    collected_tool_calls[idx]["name"] += tc_delta.function.name
                if tc_delta.function and tc_delta.function.arguments:
                    collected_tool_calls[idx]["arguments"] += tc_delta.function.arguments

    return "".join(content_chunks), list(collected_tool_calls.values())


text, tool_calls = stream_agent_turn([{"role": "user", "content": "What's the weather in New York?"}])
print("Streamed content:", text)
print("Assembled tool calls:", tool_calls)


Streamed content: 
Assembled tool calls: [{'id': 'call_Ef9TZz7FySf0Gdxs1J8tj3pk', 'name': 'get_weather', 'arguments': '{"city":"New York"}'}]
